# LASSO / Elastic Net Feature Selection (LOOCV)

Another simple alternative to `Feature_Weighting.ipynb`. Runs both pure LASSO
(L1-penalized logistic regression, `l1_ratio=1.0`) and Elastic Net
(`l1_ratio` between 0 and 1) side by side, leave-one-out cross-validated (8
folds) per sex — same pattern as `RandomForest_Feature_Importance.ipynb`.

LASSO's L1 penalty zeroes out most gene coefficients, but it arbitrarily
picks just one gene from a group of correlated genes and drops the rest.
Elastic Net tends to keep or drop correlated genes together instead — worth
comparing directly since real correlated gene pairs turned up among the top
attribute-weighting hits (e.g. Dbp-Ky r=0.95).

Both tables also get a `HedgesG` column — a small-sample-corrected effect
size computed directly from expression values, independent of either model,
so it's directly comparable between the two.

**Real data guard:** `USE_REAL_DATA` must stay `False` in this AI session. Only set `True` for your own local, offline run (see `CLAUDE.md`).

**Limitations:**
- How many genes get picked depends a lot on one setting, `C` (regularization strength) — the sweep steps below find a sane value for each method instead of guessing. LASSO and Elastic Net need separate sweeps — the same `C` means something different at a different `l1_ratio`.
- Only 8 samples, thousands of genes — treat this as a cross-check, not a replacement for the main pipeline.
- Male and female are kept separate.

In [ ]:
USE_REAL_DATA = True # keep False here. Only set True in your own local run, never inside an AI tool session, per CLAUDE.md.
RUN_ELASTIC_NET = True # Elastic Net now included, for a larger pathway-enrichment-sized gene set (LASSO's shortlist is too small for enrichment)
RUN_PERMUTATION_TEST = True # Step 7 is slow (~10 min combined) and stable once already checked — set False to skip it on reruns

import time
import pandas as pd

from lasso_selection import run_loocv_lasso, sweep_C, hedges_g, cpm_log_normalize

SEXES = ["female", "male"]
EXPECTED_GENE_COUNTS = {"female": 2881, "male": 4257}
DATA_DIR = "restructured"
TOP_N = 10
ELASTIC_NET_L1_RATIO = 0.5  # 1.0 = pure LASSO, 0.0 = pure Ridge, in between = Elastic Net

## Step 1. Load data and check shapes

In [ ]:
data = {}

for sex in SEXES:
    if USE_REAL_DATA:
        df = pd.read_csv(f"{DATA_DIR}/{sex}_expr.csv")
        with open(f"{DATA_DIR}/{sex}_filtered_gene_symbols.txt") as f:
            gene_cols = [line.strip() for line in f if line.strip()]
    else:
        df = pd.read_csv(f"{DATA_DIR}/{sex}_expr_synthetic.csv")
        gene_cols = [c for c in df.columns if c not in ("Sample", "Group", "Label")]

    assert df.shape[0] == 8, f"{sex}: expected 8 samples, got {df.shape[0]}"
    assert len(gene_cols) == EXPECTED_GENE_COUNTS[sex], (
        f"{sex}: expected {EXPECTED_GENE_COUNTS[sex]} genes, got {len(gene_cols)}"
    )
    assert (df["Group"].value_counts() == 4).all(), f"{sex}: expected a 4/4 Group split"
    assert (df["Label"] == (df["Group"] == "C26").astype(int)).all(), (
        f"{sex}: Label does not match Group for some samples"
    )

    # CPM + log2 normalize before any feature selection — raw counts carry each
    # sample's sequencing depth as a confound (our own library sizes vary by
    # ~20% sample to sample), which the within-fold StandardScaler does not
    # correct for since it scales per gene, not per sample. Found this changes
    # which genes LASSO picks, not just a formality.
    X_normalized = cpm_log_normalize(df[gene_cols])

    data[sex] = {"X": X_normalized, "y": df["Label"]}
    print(f"{sex}: OK, {df.shape[0]} samples x {len(gene_cols)} genes, 4/4 Group split, Label matches Group, CPM+log2 normalized")

## Step 2. Sweep C for LASSO (l1_ratio=1.0)

`C` controls how many genes survive. Too small and the model selects 0 genes (gives up and predicts the majority class); too large and it selects hundreds. Look for a `C` with a small `mean_genes_per_fold` (aim for a couple dozen) and good accuracy, then set `CHOSEN_C_LASSO` below.

In [ ]:
C_GRID = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5, 0.7, 1.0]

for sex in SEXES:
    print(f"--- {sex} (LASSO, l1_ratio=1.0) ---")
    print(sweep_C(data[sex]["X"], data[sex]["y"], C_GRID, l1_ratio=1.0))
    print()

## Step 2.5. Sweep C for Elastic Net (l1_ratio=0.5)

Same idea, but a different `C` — Elastic Net's mix of L1/L2 changes how many genes
survive at a given `C`, so LASSO's sweep result above doesn't transfer. Set
`CHOSEN_C_ELASTIC_NET` below from this sweep, not from Step 2's.

In [ ]:
if RUN_ELASTIC_NET:
    for sex in SEXES:
        print(f"--- {sex} (Elastic Net, l1_ratio={ELASTIC_NET_L1_RATIO}) ---")
        print(sweep_C(data[sex]["X"], data[sex]["y"], C_GRID, l1_ratio=ELASTIC_NET_L1_RATIO))
        print()
else:
    print("RUN_ELASTIC_NET is False — skipping Elastic Net C sweep. Set RUN_ELASTIC_NET=True above to include it.")

## Step 3. Sanity check on toy data, using the chosen C for each method

Set `CHOSEN_C_LASSO` and `CHOSEN_C_ELASTIC_NET` from Steps 2 and 2.5 (0.3/0.3 worked on synthetic data — pick your own from the sweeps above). One signal gene, four noise genes — the signal gene should be the only one selected, for both methods.

In [ ]:
import numpy as np

CHOSEN_C_LASSO = 1.0  # from the Step 2 sweep on normalized data: ~25.6/37.75 genes/fold, matches "couple dozen" guidance
CHOSEN_C_ELASTIC_NET = 0.3  # from the Step 2.5 sweep: ~104/144 genes/fold, sized for pathway enrichment rather than a minimal signature

toy_y = pd.Series([0, 0, 0, 0, 1, 1, 1, 1])

rng = np.random.default_rng(42)
toy_X = pd.DataFrame({
    "Signal": [5, 8, 10, 12, 120, 125, 130, 135],  # clean SHAM-low / C26-high split
    "Noise1": rng.uniform(0, 150, size=8),
    "Noise2": rng.uniform(0, 150, size=8),
    "Noise3": rng.uniform(0, 150, size=8),
    "Noise4": rng.uniform(0, 150, size=8),
})

toy_summary, toy_accuracy, _ = run_loocv_lasso(toy_X, toy_y, C=CHOSEN_C_LASSO, l1_ratio=1.0)
print(toy_summary)
print(f"toy LOOCV accuracy (LASSO): {toy_accuracy:.0%}")

assert toy_summary.index[0] == "Signal", "Signal gene should rank #1 by selection frequency"
print("PASS — signal gene ranked above all noise genes (LASSO)")

In [ ]:
if RUN_ELASTIC_NET:
    toy_summary_en, toy_accuracy_en, _ = run_loocv_lasso(toy_X, toy_y, C=CHOSEN_C_ELASTIC_NET, l1_ratio=ELASTIC_NET_L1_RATIO)
    print(toy_summary_en)
    print(f"toy LOOCV accuracy (Elastic Net): {toy_accuracy_en:.0%}")

    assert toy_summary_en.index[0] == "Signal", "Signal gene should rank #1 by selection frequency"
    print("PASS — signal gene ranked above all noise genes (Elastic Net)")
else:
    print("RUN_ELASTIC_NET is False — skipping Elastic Net toy sanity check.")

## Step 4. Run LOOCV LASSO on both sexes

Each fold fits LASSO on 7 samples and predicts the 1 held out. `SelectionFreq` is how often a gene survived with a nonzero coefficient across the 8 folds — genes at 1.0 were picked every time.

In [ ]:
summary_results = {}
accuracy_results = {}
fold_detail_results = {}

for sex in SEXES:
    t0 = time.time()
    summary, accuracy, fold_detail = run_loocv_lasso(data[sex]["X"], data[sex]["y"], C=CHOSEN_C_LASSO, l1_ratio=1.0)
    elapsed = time.time() - t0

    summary_results[sex] = summary
    accuracy_results[sex] = accuracy
    fold_detail_results[sex] = fold_detail

    n_ever_selected = (summary["SelectionFreq"] > 0).sum()
    print(f"--- {sex} (LASSO) ---  ({elapsed:.1f}s, C={CHOSEN_C_LASSO})")
    print(f"LOOCV accuracy: {accuracy:.0%} ({fold_detail['correct'].sum()}/{len(fold_detail)})")
    print(f"mean genes selected per fold: {fold_detail['n_genes_selected'].mean():.1f}")
    print(f"genes ever selected across folds: {n_ever_selected}")
    print(f"Top {TOP_N} genes by SelectionFreq:")
    print(summary.head(TOP_N))
    print()

## Step 4.5. Run LOOCV Elastic Net on both sexes

Same idea as Step 4, `l1_ratio=ELASTIC_NET_L1_RATIO` instead of pure LASSO.

In [ ]:
summary_results_en = {}
accuracy_results_en = {}
fold_detail_results_en = {}

if RUN_ELASTIC_NET:
    for sex in SEXES:
        t0 = time.time()
        summary, accuracy, fold_detail = run_loocv_lasso(
            data[sex]["X"], data[sex]["y"], C=CHOSEN_C_ELASTIC_NET, l1_ratio=ELASTIC_NET_L1_RATIO
        )
        elapsed = time.time() - t0

        summary_results_en[sex] = summary
        accuracy_results_en[sex] = accuracy
        fold_detail_results_en[sex] = fold_detail

        n_ever_selected = (summary["SelectionFreq"] > 0).sum()
        print(f"--- {sex} (Elastic Net) ---  ({elapsed:.1f}s, C={CHOSEN_C_ELASTIC_NET}, l1_ratio={ELASTIC_NET_L1_RATIO})")
        print(f"LOOCV accuracy: {accuracy:.0%} ({fold_detail['correct'].sum()}/{len(fold_detail)})")
        print(f"mean genes selected per fold: {fold_detail['n_genes_selected'].mean():.1f}")
        print(f"genes ever selected across folds: {n_ever_selected}")
        print(f"Top {TOP_N} genes by SelectionFreq:")
        print(summary.head(TOP_N))
        print()
else:
    print("RUN_ELASTIC_NET is False — skipping Elastic Net LOOCV. summary_results_en etc. left empty.")

## Step 4.6. Attach Hedges' g effect size to both tables

Computed once per sex directly from expression values (independent of either
model), then joined onto both the LASSO and Elastic Net summary tables — so
the same gene's effect size is directly comparable between the two methods.

In [ ]:
effect_sizes = {}

for sex in SEXES:
    g = hedges_g(data[sex]["X"], data[sex]["y"])
    effect_sizes[sex] = g

    summary_results[sex] = summary_results[sex].join(g.rename("HedgesG"))
    if RUN_ELASTIC_NET:
        summary_results_en[sex] = summary_results_en[sex].join(g.rename("HedgesG"))

    print(f"--- {sex}: top {TOP_N} by |HedgesG| ---")
    print(g.reindex(g.abs().sort_values(ascending=False).index).head(TOP_N).rename("HedgesG"))
    print()

## Step 4.7. Build a pathway-enrichment-sized gene set (Elastic Net only)

LASSO's `SelectionFreq == 1.0` shortlist (Step 5/6) is deliberately minimal —
good for a headline signature, too small for pathway enrichment. Elastic
Net's grouping behavior (keeps correlated genes together instead of
arbitrarily dropping them) makes it the better source for a larger set.

Filtered to `SelectionFreq >= 0.25` (selected in at least 2 of the 8 LOOCV
folds) — drops the weakest single-fold-only inclusions while keeping the set
close to a "hundred or more" size, chosen after checking a few candidate `C`
values against external-cohort direction agreement and transfer accuracy
(C=0.3 was the best tradeoff at the time this was set).

In [ ]:
PATHWAY_SET_MIN_FREQ = 0.25

pathway_gene_sets = {}

if RUN_ELASTIC_NET:
    for sex in SEXES:
        ever = summary_results_en[sex][summary_results_en[sex]["SelectionFreq"] > 0]
        pathway_set = ever[ever["SelectionFreq"] >= PATHWAY_SET_MIN_FREQ].sort_values(
            ["SelectionFreq", "AbsMeanCoef"], ascending=[False, False]
        )
        pathway_gene_sets[sex] = pathway_set

        print(f"--- {sex} ---")
        print(f"genes ever selected: {len(ever)}, pathway-enrichment set (freq >= {PATHWAY_SET_MIN_FREQ}): {len(pathway_set)}")
        print(f"of those, selected in every fold: {(pathway_set['SelectionFreq'] == 1.0).sum()}")
        print()
else:
    print("RUN_ELASTIC_NET is False — skipping pathway gene set construction.")

## Step 5. Save results

Saves the full selection-frequency table (with `HedgesG`), per-fold predictions, and a shortlist (genes selected in every fold) for each sex — separately for LASSO and Elastic Net.

In [ ]:
def save_method_results(method_label, summaries, fold_details):
    for sex in SEXES:
        summary = summaries[sex]

        summary_path = f"{DATA_DIR}/{sex}_{method_label}_selection.csv"
        shortlist_path = f"{DATA_DIR}/{sex}_{method_label}_shortlist_genes.txt"
        fold_detail_path = f"{DATA_DIR}/{sex}_{method_label}_loocv_folds.csv"

        summary.to_csv(summary_path, index_label="Gene")
        fold_details[sex].to_csv(fold_detail_path, index=False)

        shortlist = summary.index[summary["SelectionFreq"] == 1.0].tolist()
        with open(shortlist_path, "w") as f:
            f.write("\n".join(shortlist))

        print(f"{sex} ({method_label}): saved {len(summary)}-gene selection table -> {summary_path}")
        print(f"{sex} ({method_label}): saved {len(shortlist)} genes selected in every fold -> {shortlist_path}")
        print(f"{sex} ({method_label}): saved {len(fold_details[sex])}-fold LOOCV detail -> {fold_detail_path}")

save_method_results("lasso", summary_results, fold_detail_results)

if RUN_ELASTIC_NET:
    print()
    save_method_results("elastic_net", summary_results_en, fold_detail_results_en)

    print()
    for sex in SEXES:
        pathway_set = pathway_gene_sets[sex]
        pathway_csv_path = f"{DATA_DIR}/{sex}_elastic_net_pathway_geneset.csv"
        pathway_txt_path = f"{DATA_DIR}/{sex}_elastic_net_pathway_geneset.txt"

        pathway_set.to_csv(pathway_csv_path, index_label="Gene")
        with open(pathway_txt_path, "w") as f:
            f.write("\n".join(pathway_set.index.tolist()))

        print(f"{sex} (elastic_net pathway set): saved {len(pathway_set)} genes (freq>={PATHWAY_SET_MIN_FREQ}) -> {pathway_csv_path}")
        print(f"{sex} (elastic_net pathway set): saved plain gene symbol list -> {pathway_txt_path}")
else:
    print("\nRUN_ELASTIC_NET is False — skipped saving Elastic Net results.")

## Step 6. Compare with attribute weighting results

Checks overlap between LASSO/Elastic Net's stable genes (`SelectionFreq == 1.0`)
and attribute weighting's stability-ranked 30-gene shortlist
(`Feature_Weighting.ipynb`, Step 6) — this is the actual point of running two
independent methods on the same data: genes both agree on are your strongest
candidates.

In [ ]:
for sex in SEXES:
    aw_path = f"{DATA_DIR}/{sex}_signature_genes_draft_shortlist.txt"
    try:
        with open(aw_path) as f:
            aw_shortlist = set(line.strip() for line in f if line.strip())
    except FileNotFoundError:
        print(f"{sex}: {aw_path} not found — run Feature_Weighting.ipynb's Step 6 first")
        continue

    lasso_shortlist = set(summary_results[sex].index[summary_results[sex]["SelectionFreq"] == 1.0])

    print(f"--- {sex} ---")
    print(f"attribute weighting shortlist: {len(aw_shortlist)} genes")
    print(f"LASSO shortlist:               {len(lasso_shortlist)} genes")
    print(f"LASSO ∩ attribute weighting:       {sorted(lasso_shortlist & aw_shortlist)}")

    if RUN_ELASTIC_NET:
        en_shortlist = set(summary_results_en[sex].index[summary_results_en[sex]["SelectionFreq"] == 1.0])
        print(f"Elastic Net shortlist:         {len(en_shortlist)} genes")
        print(f"Elastic Net ∩ attribute weighting: {sorted(en_shortlist & aw_shortlist)}")
        print(f"LASSO ∩ Elastic Net:               {sorted(lasso_shortlist & en_shortlist)}")
    print()

## Step 7. Permutation-null test for LOOCV accuracy

At n=8 with a balanced 4/4 split, there are only C(8,4)=70 possible balanced
label assignments — few enough to test all of them exactly (not an
approximation). Reruns the full LOOCV pipeline once per permutation, at the
same `C`/`l1_ratio` already chosen from the real data. `p` = fraction of all
70 permutations (including the true one) that reach accuracy at least as
high as the real labels.

**Expected runtime**: ~1-2 min for LASSO per sex, ~5-8 min for Elastic Net per
sex (Elastic Net's `saga` solver is slower per fit).

**Important**: with thousands of genes and only 7 training samples per fold,
even pure noise can hit high LOOCV accuracy by chance — this test might not
come back favorable, and that's a real result to report honestly, not a bug.

In [ ]:
from lasso_selection import permutation_test_accuracy

permutation_results = {}

if RUN_PERMUTATION_TEST:
    for sex in SEXES:
        t0 = time.time()
        obs, null_dist, p = permutation_test_accuracy(data[sex]["X"], data[sex]["y"], C=CHOSEN_C_LASSO, l1_ratio=1.0)
        elapsed = time.time() - t0
        permutation_results[("lasso", sex)] = (obs, null_dist, p)
        print(f"--- {sex} (LASSO) ---  ({elapsed:.1f}s, {len(null_dist)} permutations)")
        print(f"observed accuracy: {obs:.0%}   null mean: {null_dist.mean():.0%}   null max: {null_dist.max():.0%}   p={p:.3f}")
        print()
else:
    print("RUN_PERMUTATION_TEST is False — skipping LASSO permutation test.")

In [ ]:
if RUN_PERMUTATION_TEST and RUN_ELASTIC_NET:
    for sex in SEXES:
        t0 = time.time()
        obs, null_dist, p = permutation_test_accuracy(
            data[sex]["X"], data[sex]["y"], C=CHOSEN_C_ELASTIC_NET, l1_ratio=ELASTIC_NET_L1_RATIO
        )
        elapsed = time.time() - t0
        permutation_results[("elastic_net", sex)] = (obs, null_dist, p)
        print(f"--- {sex} (Elastic Net) ---  ({elapsed:.1f}s, {len(null_dist)} permutations)")
        print(f"observed accuracy: {obs:.0%}   null mean: {null_dist.mean():.0%}   null max: {null_dist.max():.0%}   p={p:.3f}")
        print()
elif not RUN_PERMUTATION_TEST:
    print("RUN_PERMUTATION_TEST is False — skipping Elastic Net permutation test.")
else:
    print("RUN_ELASTIC_NET is False — skipping Elastic Net permutation test.")

In [ ]:
rows = []
for (method, sex), (obs, null_dist, p) in permutation_results.items():
    rows.append({
        "method": method,
        "sex": sex,
        "observed_accuracy": obs,
        "null_mean_accuracy": null_dist.mean(),
        "null_max_accuracy": null_dist.max(),
        "n_permutations": len(null_dist),
        "p_value": p,
    })

permutation_summary = pd.DataFrame(rows)
permutation_summary_path = f"{DATA_DIR}/permutation_test_accuracy_summary.csv"
permutation_summary.to_csv(permutation_summary_path, index=False)
print(permutation_summary)
print(f"\nsaved -> {permutation_summary_path}")

## Step 8. Correlation heatmap of the final curated gene sets

Manually curated shortlists (female: the 24-gene Elastic Net subset reviewed
for the presentation; male: the analogous 19-gene subset) — not re-derived
from a formula here, just hardcoded from that review. Hierarchical clustering
on the gene-gene correlation matrix (CPM+log2 normalized, Pearson, average
linkage), styled as a WGCNA-style module plot.

**Deliberately NOT run on the full 2,881/4,257-gene candidate panel.** Every
gene in that panel was already pre-filtered for being significant
(FDR<0.05) against the same SHAM/C26 label, so a full-panel correlation
heatmap just reproduces the same up/down split as a giant checkerboard —
it looks like discovered co-expression modules but isn't really showing
anything beyond the label split you already know. The curated subset is
small enough, and specific enough, to be a genuine, honest supporting figure
instead.

In [ ]:
import seaborn as sns

CURATED_GENE_SETS = {
    "female": """Prodh
Per3
Cd163
Ppp1r3c
Disp1
Hlf
Dpy19l3
Itpripl1
Stard3
Crim1
Per2
Ank2
Grrp1
Uevld
Large1
Nrp2
Actr3
Rilpl1
Cdkn1a
Ppargc1b
Angpt1
Cldn15
Tnfrsf1b
Dbp""".split(),
    "male": """Eif2a
Tmem86a
Slc25a10
Anapc16
Grrp1
Ppp1r3c
Eno1
Cdkn1a
Rorc
Dbp
Erfe
Acbd4
Tmem176b
Tab1
Tagap
Ddx59
Abcb9
Dcakd
Dok5""".split(),
}

for sex in SEXES:
    genes = CURATED_GENE_SETS[sex]
    corr = data[sex]["X"][genes].corr(method="pearson")

    g = sns.clustermap(
        corr, cmap="RdBu_r", vmin=-1, vmax=1, method="average",
        figsize=(9, 9), cbar_kws={"label": "Corr."},
    )
    g.fig.suptitle(f"{sex} — curated gene set (n={len(genes)})", y=1.02)

    heatmap_path = f"{DATA_DIR}/{sex}_curated_geneset_heatmap.png"
    g.savefig(heatmap_path, dpi=150, bbox_inches="tight")
    print(f"{sex}: saved {len(genes)}-gene correlation heatmap -> {heatmap_path}")

### Step 8b. Reference only — full candidate panel (NOT recommended for the presentation)

Same idea, run on the entire 2,881/4,257-gene candidate panel instead of the
curated subset. Included for comparison, not as a figure to present: every
gene here already passed FDR<0.05 against the same SHAM/C26 label, so this
mostly just re-displays the up/down split as two giant blocks rather than
showing real finer co-expression structure — see the Step 8 markdown above
for the full reasoning.

**Slow**: ~30s (female, 2881 genes) to ~2 min (male, 4257 genes) to compute
the correlation matrix and cluster it.

In [ ]:
RUN_FULL_PANEL_HEATMAP = True # slow (~30s-2min per sex) and not recommended for the talk — see markdown above

if RUN_FULL_PANEL_HEATMAP:
    for sex in SEXES:
        full_corr = data[sex]["X"].corr(method="pearson")

        g_full = sns.clustermap(
            full_corr, cmap="RdBu_r", vmin=-1, vmax=1, method="average",
            figsize=(10, 10), cbar_kws={"label": "Corr."},
            xticklabels=False, yticklabels=False,
        )
        g_full.fig.suptitle(f"{sex} — full candidate panel (n={full_corr.shape[0]}, reference only)", y=1.02)

        full_heatmap_path = f"{DATA_DIR}/{sex}_full_panel_heatmap.png"
        g_full.savefig(full_heatmap_path, dpi=150, bbox_inches="tight")
        print(f"{sex}: saved {full_corr.shape[0]}-gene full-panel correlation heatmap -> {full_heatmap_path}")
else:
    print("RUN_FULL_PANEL_HEATMAP is False — skipping. Set True above to generate it (slow, reference only).")

## Step 9. External validation of the curated gene sets (GSE276018)

Fits one Elastic Net model per sex on all 8 own samples, restricted to just
the `CURATED_GENE_SETS` genes from Step 8, then tests it against an
independent cohort (GSE276018) never touched during discovery. Two checks:

- **Raw transfer accuracy** — does the fitted model correctly classify the
  external mice (fragile: depends on getting the right decision boundary,
  not just the right direction).
- **Direction agreement** — does each gene's coefficient sign match the sign
  of its Hedges' g in the external cohort (softer, less fragile check).

Requires `GSE276018_precachectic_subset_with_symbols.csv` and
`GSE276018_precachectic_sample_metadata.csv` in the project root (same files
`External_Dataset_Pooling.ipynb` uses) — gitignored, local-only.

In [ ]:
from sklearn.linear_model import LogisticRegression

EXTERNAL_GENE_EXPR_PATH = "GSE276018_precachectic_subset_with_symbols.csv"
EXTERNAL_METADATA_PATH = "GSE276018_precachectic_sample_metadata.csv"

ext_expr = pd.read_csv(EXTERNAL_GENE_EXPR_PATH, index_col="EnsemblID")
ext_meta = pd.read_csv(EXTERNAL_METADATA_PATH)
ext_all_genes = sorted(set(ext_expr["Symbol"].dropna()))

external_validation_results = {}

for sex in SEXES:
    genes = CURATED_GENE_SETS[sex]

    sex_meta = ext_meta[ext_meta["sex"] == sex]
    titles = sex_meta["title"].tolist()
    y_test = pd.Series((sex_meta["Group"] == "C26_10day").astype(int).values)

    # normalize on the FULL external gene panel before subsetting — CPM's
    # library-size denominator must be the whole panel, not just these genes
    ext_wide = ext_expr.set_index("Symbol")[titles].T
    ext_wide = ext_wide.loc[:, ~ext_wide.columns.duplicated()]
    ext_wide = ext_wide[[g for g in ext_all_genes if g in ext_wide.columns]].astype(float)
    ext_norm = cpm_log_normalize(ext_wide).reset_index(drop=True)

    common_genes = [g for g in genes if g in ext_norm.columns]
    X_train = data[sex]["X"][common_genes]
    y_train = data[sex]["y"]
    X_test = ext_norm[common_genes]

    # within-study standardization on both sides — batch effects break the
    # usual "fit scaler on train, apply to test" assumption across studies
    X_train_z = (X_train - X_train.mean()) / X_train.std(ddof=0).replace(0, 1)
    X_test_z = (X_test - X_test.mean()) / X_test.std(ddof=0).replace(0, 1)

    # no explicit `penalty=` — same style as lasso_selection.py's own model
    # fits, and avoids a version-dependent behavior difference: newer sklearn
    # (>=1.8) deprecates penalty='elasticnet' alongside l1_ratio in favor of
    # l1_ratio alone, and mixing the two styles gave different coefficients
    # (and a different external accuracy) depending on sklearn version.
    clf = LogisticRegression(
        l1_ratio=ELASTIC_NET_L1_RATIO, C=CHOSEN_C_ELASTIC_NET,
        random_state=51, solver="saga", max_iter=5000,
    )
    clf.fit(X_train_z.values, y_train)
    preds = clf.predict(X_test_z.values)
    accuracy = (preds == y_test.values).mean()
    baseline = max(y_test.mean(), 1 - y_test.mean())

    own_coefs = pd.Series(clf.coef_[0], index=common_genes)
    nonzero = own_coefs[own_coefs != 0]
    ext_g = hedges_g(ext_norm[nonzero.index], y_test)
    agree = (np.sign(nonzero) == np.sign(ext_g))

    external_validation_results[sex] = {
        "accuracy": accuracy, "baseline": baseline,
        "n_genes_used": len(nonzero), "n_genes_total": len(common_genes),
        "direction_agree": agree.sum(), "direction_total": len(agree),
    }

    print(f"--- {sex} ---")
    print(f"genes present in external data: {len(common_genes)}/{len(genes)}")
    print(f"genes with nonzero coefficient (not dropped by L1): {len(nonzero)}/{len(common_genes)}")
    print(f"raw transfer accuracy: {accuracy:.0%} (majority baseline: {baseline:.0%}, n={len(y_test)})")
    print(f"direction agreement: {agree.sum()}/{len(agree)} ({agree.mean():.0%})")
    print()

In [ ]:
external_validation_summary = pd.DataFrame(external_validation_results).T
external_validation_summary.index.name = "sex"
external_validation_path = f"{DATA_DIR}/curated_geneset_external_validation.csv"
external_validation_summary.to_csv(external_validation_path)
print(external_validation_summary)
print(f"\nsaved -> {external_validation_path}")